# Praktik Pengolahan Citra Digital
**Segmentasi Simbol Sekop ♠, Hati ♥, Wajik ♦, dan Keriting ♣ pada Kartu Remi**

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Import modul pengolah_citra
from pengolah_citra.langkah1_grayscale import proses_grayscale
from pengolah_citra.langkah2_geometri import proses_geometri
from pengolah_citra.langkah3_tepi import deteksi_sobel, deteksi_prewitt
from pengolah_citra.langkah4_segmentasi import proses_segmentasi

def show_images(images, titles, max_cols=3):
    n = len(images)
    cols = min(n, max_cols)
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(5 * cols, 5 * rows))
    for i in range(n):
        plt.subplot(rows, cols, i + 1)
        if len(images[i].shape) == 3:
            plt.imshow(images[i])
        else:
            plt.imshow(images[i], cmap='gray')
        plt.title(titles[i])
        plt.axis('off')
    plt.tight_layout()
    plt.show()

### Langkah 1: Upload dan Konversi Grayscale
Memuat gambar, mengubah ke grayscale, serta mengatur Brightness dan Contrast.

In [ ]:
brightness = 0
contrast = 1.0

# === PAKAI MODUL ===
img_gray_adjusted = proses_grayscale(img_gray, brightness=brightness, contrast=contrast)

show_images(
    [img_rgb, img_gray_adjusted],
    ['Citra Asli (RGB)', f'Grayscale Adjusted (B={brightness}, C={contrast})']
)

### Langkah 2: Operasi Geometri Citra
Hanya menggunakan Cropping dan Flipping berdasarkan instruksi yang diperbarui.

In [ ]:
# Input dari Langkah 1
img_gray_input = img_gray_adjusted.copy()
img_rgb_input = img_rgb.copy()

h, w = img_gray_input.shape[:2]

# Pengaturan Geometri
# 1. Cropping
start_y, end_y = 0, h
start_x, end_x = 0, w

if start_y < end_y and start_x < end_x:
    img_gray_input = img_gray_input[start_y:end_y, start_x:end_x]
    img_rgb_input = img_rgb_input[start_y:end_y, start_x:end_x]

# 2. Flipping (0: Vertikal, 1: Horizontal, -1: Keduanya, None: Tidak ada)
flip_mode = None 

if flip_mode is not None:
    img_gray_input = cv2.flip(img_gray_input, flip_mode)
    img_rgb_input = cv2.flip(img_rgb_input, flip_mode)

# Simpan output ke variabel output langkah 2
img_gray_geo = img_gray_input
img_rgb_geo = img_rgb_input

show_images([img_rgb_geo, img_gray_geo], ['RGB Geometri', 'Grayscale Geometri'])

### Langkah 3: Deteksi Tepi Dinamis
Pilih salah satu metode deteksi tepi (Sobel ATAU Prewitt). Hasil garis tepi ini akan **diteruskan menjadi input Tahap 4**.

In [ ]:
gray_edge_input = img_gray_geo

ksize       = 3    # ukuran kernel Sobel: 3, 5, atau 7
edge_thresh = 0    # threshold noise (0 = tampilkan semua tepi)

# === PAKAI MODUL ===
sobel_combined = deteksi_sobel(gray_edge_input, ksize=ksize, edge_thresh=edge_thresh)
prewitt_combined = deteksi_prewitt(gray_edge_input, edge_thresh=edge_thresh)

show_images(
    [sobel_combined, prewitt_combined],
    [f'Deteksi Tepi: Sobel (kernel={ksize}, threshold={edge_thresh})',
     f'Deteksi Tepi: Prewitt (threshold={edge_thresh})']
)

### Langkah 4: Segmentasi dari Garis Tepi
Mengambil gambar **garis tepi** dari Langkah 3, lalu menambal (fill) konturnya menjadi area solid, dan mengekstrak warna simbol dari RGB asli.

In [ ]:
gray_seg_input = img_gray_geo
rgb_seg_input  = img_rgb_geo

use_otsu       = True   
manual_thresh  = 127    
use_inverse    = False  
min_area       = 50     
max_area       = 15000  

# === PAKAI MODUL ===
ret, thresh_img, symbol_count, img_segmented_symbols = proses_segmentasi(
    gray_seg_input, rgb_seg_input,
    use_otsu=use_otsu, manual_thresh=manual_thresh, use_inverse=use_inverse,
    min_area=min_area, max_area=max_area
)

print(f'Threshold digunakan: {ret}')
print(f'Kontur simbol terdeteksi: {symbol_count}')

show_images(
    [thresh_img, img_segmented_symbols],
    [f'Threshold={ret:.0f} | {"Inverse" if use_inverse else "Normal"}',
     f'Segmentasi Simbol ({symbol_count} kontur)']
)